# Visión Computacional: de los píxeles al pipeline de aumentación

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ecamposv/nlp-vision/blob/main/semana-05/Tema-Integrador/VisionComputacional_Integrador.ipynb)

**Notebook integrador (Tema 9 + Tema 10) — nivel maestría**

Este notebook combina y profundiza los conceptos de los dos temas de la semana 5. A diferencia de los notebooks individuales, aquí cada bloque tiene:

- **Fundamento matemático** (formato KaTeX).
- **Implementación con OpenCV / NumPy / Albumentations**.
- **Widget interactivo** (`ipywidgets`) para explorar parámetros en tiempo real.
- **Discusión de cuándo y por qué** usar cada técnica en producción.

> **Objetivos de aprendizaje.** Al terminar serás capaz de: (1) describir matemáticamente una imagen digital y elegir el espacio de color adecuado para una tarea; (2) construir y justificar un pipeline de preprocesamiento para una red neuronal; (3) implementar aumentación de datos profesional con `Albumentations`; (4) cuantificar la calidad de un filtro de denoising con PSNR.

---

## 0. Setup (Colab y local)

Esta celda:

1. Instala `opencv-python`, `albumentations` e `ipywidgets` si faltan.
2. Descarga la imagen `tigre.png` desde el repositorio si no existe localmente.
3. Configura `matplotlib` para gráficos consistentes y de buena resolución.

> En **Google Colab** los widgets requieren habilitar el *custom widget manager* — esto se hace automáticamente al final de la celda.

In [ ]:
import os
import sys
import subprocess
import urllib.request


def _ensure(paquete, import_name=None):
    """Instala un paquete con pip si no está disponible."""
    nombre = import_name or paquete
    try:
        __import__(nombre)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", paquete])


_ensure("opencv-python", "cv2")
_ensure("albumentations")
_ensure("ipywidgets")

REPO_RAW = "https://raw.githubusercontent.com/ecamposv/nlp-vision/main/semana-05/Tema-10"
IMG = "tigre.png"

if not os.path.exists(IMG):
    urllib.request.urlretrieve(f"{REPO_RAW}/{IMG}", IMG)
    print(f"[Descargado] {IMG} ({os.path.getsize(IMG)} bytes)")
else:
    print(f"[OK] {IMG} ya existe ({os.path.getsize(IMG)} bytes)")

# Habilitar widgets en Colab (no afecta si no es Colab)
try:
    from google.colab import output as _colab_output  # type: ignore
    _colab_output.enable_custom_widget_manager()
    print("[Colab] Widget manager habilitado")
except ImportError:
    pass

import matplotlib.pyplot as plt
plt.rcParams["figure.dpi"] = 110
plt.rcParams["savefig.dpi"] = 140
plt.rcParams["image.interpolation"] = "nearest"
print("Setup completo.")


## 1. La imagen digital: muestreo, cuantización y tipos de dato

Una imagen digital en color es una **función discreta**

$$I : \{0,\dots,H-1\} \times \{0,\dots,W-1\} \to \{0,\dots,255\}^3$$

que asigna a cada coordenada $(y, x)$ un vector de **3 canales**. En memoria es un arreglo NumPy con forma `(H, W, C)` y tipo `uint8` por defecto en OpenCV.

Conceptos clave:

| Concepto | Descripción |
|---|---|
| **Muestreo (sampling)** | Discretización espacial: define $H \times W$. |
| **Cuantización** | Discretización de intensidad: `uint8` usa 8 bits/canal (256 niveles). |
| **Profundidad de bits** | `uint8` (estándar), `uint16` (médico/RAW), `float32` (modelos de DL). |
| **Orden de canales** | OpenCV usa **BGR**; matplotlib / PIL / PyTorch usan **RGB**. |

**Por qué importa.** Confundir BGR/RGB es la causa #1 de "el modelo entrenó bien pero predice raro". Confundir `uint8 [0,255]` con `float32 [0,1]` es la causa #2.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

img_bgr = cv2.imread("tigre.png")
if img_bgr is None:
    raise FileNotFoundError("Ejecuta primero la celda de Setup.")

img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
H, W, C = img_rgb.shape

print(f"Forma (H, W, C): {img_rgb.shape}")
print(f"dtype: {img_rgb.dtype}  |  rango: [{img_rgb.min()}, {img_rgb.max()}]")
print(f"Tamaño en memoria: {img_rgb.nbytes / 1024:.1f} KB")
print(f"Megapíxeles: {H * W / 1e6:.2f} MP")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].imshow(img_rgb)
axes[0].set_title(f"Imagen completa  {W}x{H}")
axes[0].axis("off")

# Zoom a un parche de 16x16 para ver los píxeles
y0, x0 = H // 2, W // 2
patch = img_rgb[y0:y0 + 16, x0:x0 + 16]
axes[1].imshow(patch)
axes[1].set_title("Zoom 16x16 px: cada cuadro es un píxel")
axes[1].axis("off")
# anotar valores RGB de los píxeles
for i in range(16):
    for j in range(16):
        r, g, b = patch[i, j]
        # Texto solo si la celda es lo bastante visible (omitir para no saturar)
        if (i + j) % 4 == 0:
            axes[1].text(j, i, f"{r}\n{g}\n{b}", ha="center", va="center",
                         fontsize=5, color="white" if patch[i, j].mean() < 128 else "black")
plt.tight_layout()
plt.show()


## 2. Espacios de color

Cada espacio de color reorganiza la información cromática para resaltar cierta propiedad:

| Espacio | Modelo | Cuándo usarlo |
|---|---|---|
| **RGB / BGR** | Aditivo (mezcla de luz). | Captura, almacenamiento, modelos preentrenados. |
| **HSV** | Cilíndrico: matiz, saturación, valor. | Segmentación por color, ajustes de iluminación. |
| **Lab** ($L^*a^*b^*$) | Perceptual: $L$ = luminancia, $a$ = verde-rojo, $b$ = azul-amarillo. | Comparación perceptual de color, *color transfer*. |
| **YCrCb** | Luminancia + crominancia. | Compresión (JPEG), ecualización sin alterar color. |

La separación **luminancia / crominancia** (Lab, YCrCb) permite manipular el contraste sin desbalancear los colores — algo que ecualizar directamente en RGB **sí** rompe.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

img_bgr = cv2.imread("tigre.png")
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
img_hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
img_lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2Lab)
img_ycc = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2YCrCb)

espacios = [
    ("RGB", img_rgb,                       ["R", "G", "B"]),
    ("HSV", img_hsv,                       ["H", "S", "V"]),
    ("Lab", img_lab,                       ["L", "a", "b"]),
    ("YCrCb", img_ycc,                     ["Y", "Cr", "Cb"]),
]

fig, axes = plt.subplots(len(espacios), 4, figsize=(14, 13))
for fila, (nombre, im, etiquetas) in enumerate(espacios):
    # Columna 0: imagen reconstruida en RGB para visualizar
    if nombre == "RGB":
        vis = im
    elif nombre == "HSV":
        vis = cv2.cvtColor(im, cv2.COLOR_HSV2RGB)
    elif nombre == "Lab":
        vis = cv2.cvtColor(im, cv2.COLOR_Lab2RGB)
    else:  # YCrCb
        vis = cv2.cvtColor(im, cv2.COLOR_YCrCb2RGB)
    axes[fila, 0].imshow(vis)
    axes[fila, 0].set_title(f"{nombre} (vista)")
    axes[fila, 0].axis("off")
    # Columnas 1-3: canales individuales
    for c in range(3):
        axes[fila, c + 1].imshow(im[..., c], cmap="gray")
        axes[fila, c + 1].set_title(f"{nombre} : canal {etiquetas[c]}")
        axes[fila, c + 1].axis("off")
plt.tight_layout()
plt.show()


### 2.1. Explorador interactivo de canales

Selecciona un espacio y un canal con los sliders. Mira en qué espacio el **pelaje del tigre** se separa mejor del fondo — esa es justamente la propiedad que aprovecha la segmentación por color.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, Dropdown, IntSlider

img_bgr = cv2.imread("tigre.png")

_ESPACIOS = {
    "RGB":   (cv2.COLOR_BGR2RGB,   ["R", "G", "B"]),
    "HSV":   (cv2.COLOR_BGR2HSV,   ["H", "S", "V"]),
    "Lab":   (cv2.COLOR_BGR2Lab,   ["L", "a", "b"]),
    "YCrCb": (cv2.COLOR_BGR2YCrCb, ["Y", "Cr", "Cb"]),
}


def explorar(espacio, canal):
    cvt, nombres = _ESPACIOS[espacio]
    im = cv2.cvtColor(img_bgr, cvt)
    plt.figure(figsize=(6, 5))
    plt.imshow(im[..., canal], cmap="gray")
    plt.title(f"{espacio} - canal {nombres[canal]} (idx={canal})")
    plt.axis("off")
    plt.show()


interact(explorar,
         espacio=Dropdown(options=list(_ESPACIOS), value="HSV", description="Espacio:"),
         canal=IntSlider(min=0, max=2, value=0, description="Canal:"));


## 3. Histogramas, ecualización y CLAHE

El **histograma** $h(k)$ cuenta cuántos píxeles tienen intensidad $k$. La **ecualización** busca aplanarlo redistribuyendo intensidades a través de la función de distribución acumulada (CDF):

$$T(k) = \mathrm{round}\!\left(\frac{(L-1)}{N} \sum_{j=0}^{k} h(j)\right)$$

donde $L = 256$ y $N = H \cdot W$.

**Problema de `cv2.equalizeHist`:** es **global** — si la imagen tiene zonas muy oscuras y muy claras, ecualizar una "rompe" la otra. **CLAHE** (*Contrast Limited Adaptive Histogram Equalization*) lo resuelve aplicando ecualización por bloques con un *clip limit* que evita amplificar ruido.

**Cuándo usar qué:**

- `cv2.equalizeHist` → imágenes con iluminación uniforme.
- `CLAHE` → imágenes médicas, fotos con sombras fuertes, mejora local.
- **Siempre** ecualizar sobre el canal de **luminancia** (Y de YCrCb, o L de Lab), nunca sobre R/G/B por separado.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

img_bgr = cv2.imread("tigre.png")
gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
ycc = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2YCrCb)

# Ecualización global
gray_eq = cv2.equalizeHist(gray)

# Ecualización del canal Y (correcto para color)
ycc_eq = ycc.copy()
ycc_eq[..., 0] = cv2.equalizeHist(ycc_eq[..., 0])
color_eq_rgb = cv2.cvtColor(cv2.cvtColor(ycc_eq, cv2.COLOR_YCrCb2BGR), cv2.COLOR_BGR2RGB)

# CLAHE sobre canal Y
clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))
ycc_clahe = ycc.copy()
ycc_clahe[..., 0] = clahe.apply(ycc_clahe[..., 0])
color_clahe_rgb = cv2.cvtColor(cv2.cvtColor(ycc_clahe, cv2.COLOR_YCrCb2BGR), cv2.COLOR_BGR2RGB)

img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes[0, 0].imshow(img_rgb);          axes[0, 0].set_title("Original");                  axes[0, 0].axis("off")
axes[0, 1].imshow(color_eq_rgb);     axes[0, 1].set_title("Ecualización global (Y)");   axes[0, 1].axis("off")
axes[0, 2].imshow(color_clahe_rgb);  axes[0, 2].set_title("CLAHE (Y, clip=2.5)");       axes[0, 2].axis("off")

# Histogramas de luminancia
axes[1, 0].hist(ycc[..., 0].ravel(),       bins=64, range=(0, 255), color="gray")
axes[1, 0].set_title("Hist. Y original")
axes[1, 1].hist(ycc_eq[..., 0].ravel(),    bins=64, range=(0, 255), color="gray")
axes[1, 1].set_title("Hist. Y ecualizado")
axes[1, 2].hist(ycc_clahe[..., 0].ravel(), bins=64, range=(0, 255), color="gray")
axes[1, 2].set_title("Hist. Y CLAHE")
for ax in axes[1]:
    ax.set_xlabel("Intensidad"); ax.set_ylabel("Frecuencia")
plt.tight_layout()
plt.show()


### 3.1. CLAHE interactivo

Juega con `clipLimit` (cuánto se permite amplificar el contraste local) y `tileGridSize` (tamaño de las regiones). Verás:

- `clipLimit` muy alto → aparece **ruido y artefactos**.
- `tileGridSize` muy pequeño → contraste local exagerado.
- `tileGridSize` muy grande → se parece a la ecualización global.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider

img_bgr = cv2.imread("tigre.png")
ycc_base = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2YCrCb)


def aplicar_clahe(clip_limit, tile):
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=(tile, tile))
    ycc = ycc_base.copy()
    ycc[..., 0] = clahe.apply(ycc[..., 0])
    rgb = cv2.cvtColor(cv2.cvtColor(ycc, cv2.COLOR_YCrCb2BGR), cv2.COLOR_BGR2RGB)
    fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
    ax[0].imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)); ax[0].set_title("Original"); ax[0].axis("off")
    ax[1].imshow(rgb); ax[1].set_title(f"CLAHE clip={clip_limit:.1f} tile={tile}x{tile}"); ax[1].axis("off")
    plt.tight_layout(); plt.show()


interact(aplicar_clahe,
         clip_limit=FloatSlider(min=0.5, max=10.0, step=0.5, value=2.5, description="clipLimit"),
         tile=IntSlider(min=2, max=32, step=2, value=8, description="tileGrid"));


## 4. Filtrado lineal, gradientes y bordes

El filtrado lineal es una **convolución** $g = f * k$ donde $k$ es un kernel. Distintos kernels resaltan distintas propiedades:

- **Promedio / Gaussiano** → suaviza (paso bajo).
- **Sobel** $G_x$, $G_y$ → derivadas direccionales (gradiente).
- **Laplaciano** $\nabla^2 f$ → curvatura (paso alto, detecta bordes y ruido).

La **magnitud del gradiente** $\|\nabla f\| = \sqrt{G_x^2 + G_y^2}$ resume "cuánto cambia" la imagen en cada punto.

**Canny** es el detector clásico: (1) suaviza con Gaussiano, (2) calcula gradiente, (3) supresión no-máxima en la dirección del gradiente, (4) histéresis con dos umbrales $T_{\text{low}} < T_{\text{high}}$ para enlazar bordes.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

img_bgr = cv2.imread("tigre.png")
gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
gray = cv2.resize(gray, (512, 512))

# Suavizado previo para reducir ruido en derivadas
blur = cv2.GaussianBlur(gray, (5, 5), 1.0)

# Sobel x, y y magnitud
sx = cv2.Sobel(blur, cv2.CV_32F, 1, 0, ksize=3)
sy = cv2.Sobel(blur, cv2.CV_32F, 0, 1, ksize=3)
mag = np.sqrt(sx**2 + sy**2)
mag_u8 = np.clip(mag / mag.max() * 255.0, 0, 255).astype(np.uint8)

# Laplaciano
lap = cv2.Laplacian(blur, cv2.CV_32F, ksize=3)
lap_u8 = np.clip(np.abs(lap) / np.abs(lap).max() * 255.0, 0, 255).astype(np.uint8)

# Canny (dos umbrales)
canny = cv2.Canny(blur, 80, 180)

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, im, t in zip(
    axes.flat,
    [gray, blur, np.abs(sx).astype(np.uint8), np.abs(sy).astype(np.uint8), lap_u8, canny],
    ["Original (gris)", "GaussianBlur 5x5", "|Sobel x|", "|Sobel y|", "|Laplaciano|", "Canny (80,180)"],
):
    ax.imshow(im, cmap="gray"); ax.set_title(t); ax.axis("off")
plt.tight_layout()
plt.show()


### 4.1. Canny interactivo

Ajusta los dos umbrales y observa la **regla práctica**: $T_{\text{high}} \approx 2 \cdot T_{\text{low}}$ funciona bien para la mayoría de imágenes.

- $T_{\text{low}}$ muy bajo → muchos bordes espurios (ruido).
- $T_{\text{high}}$ muy alto → bordes fragmentados, se pierde estructura.

In [ ]:
import cv2
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider

img_bgr = cv2.imread("tigre.png")
gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
gray = cv2.resize(gray, (512, 512))
blur = cv2.GaussianBlur(gray, (5, 5), 1.0)


def canny_interactivo(t_low, t_high):
    if t_high <= t_low:
        t_high = t_low + 1
    edges = cv2.Canny(blur, t_low, t_high)
    fig, ax = plt.subplots(1, 2, figsize=(10, 4.5))
    ax[0].imshow(gray, cmap="gray"); ax[0].set_title("Original gris"); ax[0].axis("off")
    ax[1].imshow(edges, cmap="gray"); ax[1].set_title(f"Canny ({t_low}, {t_high})"); ax[1].axis("off")
    plt.tight_layout(); plt.show()


interact(canny_interactivo,
         t_low=IntSlider(min=0, max=255, step=5, value=80, description="T_low"),
         t_high=IntSlider(min=0, max=255, step=5, value=180, description="T_high"));


## 5. Segmentación por color en HSV (interactivo)

Para aislar un objeto por color usamos un **umbral por rangos** en HSV:

$$M(y, x) = \mathbb{1}\big[H_{\min} \le H \le H_{\max}\big] \cdot \mathbb{1}\big[S_{\min} \le S \le S_{\max}\big] \cdot \mathbb{1}\big[V_{\min} \le V \le V_{\max}\big]$$

OpenCV usa **$H \in [0, 179]$** (grados / 2) y $S, V \in [0, 255]$.

Para el **naranja del tigre** un buen punto de partida es $H \in [5, 25]$, $S \ge 50$, $V \ge 50$.

> **Tip de producción:** después del `inRange` aplica `cv2.morphologyEx(mask, MORPH_OPEN)` para quitar ruido y `MORPH_CLOSE` para cerrar huecos.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntRangeSlider

img_bgr = cv2.imread("tigre.png")
img_hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)


def segmentar(h_range, s_range, v_range):
    low = np.array([h_range[0], s_range[0], v_range[0]], dtype=np.uint8)
    high = np.array([h_range[1], s_range[1], v_range[1]], dtype=np.uint8)
    mask = cv2.inRange(img_hsv, low, high)
    # Limpieza morfológica
    kernel = np.ones((3, 3), np.uint8)
    mask_clean = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)
    mask_clean = cv2.morphologyEx(mask_clean, cv2.MORPH_CLOSE, kernel, iterations=2)
    seg = cv2.bitwise_and(img_rgb, img_rgb, mask=mask_clean)
    fig, ax = plt.subplots(1, 3, figsize=(14, 4.5))
    ax[0].imshow(img_rgb); ax[0].set_title("Original"); ax[0].axis("off")
    ax[1].imshow(mask_clean, cmap="gray"); ax[1].set_title(f"Máscara H={h_range} S={s_range} V={v_range}"); ax[1].axis("off")
    ax[2].imshow(seg); ax[2].set_title("Imagen segmentada"); ax[2].axis("off")
    plt.tight_layout(); plt.show()


interact(segmentar,
         h_range=IntRangeSlider(value=[5, 25],   min=0, max=179, step=1, description="H"),
         s_range=IntRangeSlider(value=[50, 255], min=0, max=255, step=5, description="S"),
         v_range=IntRangeSlider(value=[50, 255], min=0, max=255, step=5, description="V"));


## 6. Preprocesamiento para modelos de Deep Learning

Toda red CNN espera un tensor con forma **(N, C, H, W)** (PyTorch) o **(N, H, W, C)** (TensorFlow), en `float32`, normalizado. Las cuatro decisiones de diseño son:

1. **Tamaño de entrada** — fijo por arquitectura (224, 256, 384, ...).
2. **Estrategia de redimensionado** — `resize` (deforma), `center crop` (recorta), `letterbox` (rellena, conserva *aspect ratio*).
3. **Interpolación** — `INTER_AREA` para reducir, `INTER_CUBIC` / `INTER_LANCZOS4` para ampliar.
4. **Normalización** — escala $[0, 1]$ o estandarización por media/std (típicamente ImageNet).

Estadísticas ImageNet (RGB, en $[0, 1]$):

$$\boldsymbol{\mu} = (0.485, 0.456, 0.406), \quad \boldsymbol{\sigma} = (0.229, 0.224, 0.225)$$

$$x_{\text{norm}} = \frac{x/255 - \boldsymbol{\mu}}{\boldsymbol{\sigma}}$$

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

img_bgr = cv2.imread("tigre.png")
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
h, w = img_rgb.shape[:2]
T = 224  # tamaño objetivo


def resize_simple(im, t):
    return cv2.resize(im, (t, t), interpolation=cv2.INTER_AREA)


def center_crop(im, t):
    h, w = im.shape[:2]
    scale = max(t / h, t / w)
    nh, nw = int(round(h * scale)), int(round(w * scale))
    im_s = cv2.resize(im, (nw, nh), interpolation=cv2.INTER_AREA if scale < 1 else cv2.INTER_CUBIC)
    y0, x0 = (nh - t) // 2, (nw - t) // 2
    return im_s[y0:y0 + t, x0:x0 + t]


def letterbox(im, t, pad=(114, 114, 114)):
    h, w = im.shape[:2]
    scale = min(t / h, t / w)
    nh, nw = int(round(h * scale)), int(round(w * scale))
    im_s = cv2.resize(im, (nw, nh), interpolation=cv2.INTER_AREA if scale < 1 else cv2.INTER_CUBIC)
    canvas = np.full((t, t, 3), pad, dtype=im.dtype)
    y0, x0 = (t - nh) // 2, (t - nw) // 2
    canvas[y0:y0 + nh, x0:x0 + nw] = im_s
    return canvas


r_simple = resize_simple(img_rgb, T)
r_crop = center_crop(img_rgb, T)
r_letter = letterbox(img_rgb, T)

# Normalización ImageNet
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)
x01 = r_letter.astype(np.float32) / 255.0
x_norm = (x01 - MEAN) / STD

# Para visualizar la versión normalizada, des-normalizamos a [0,1] solo para mostrar
x_show = np.clip(x_norm * STD + MEAN, 0, 1)

fig, axes = plt.subplots(1, 4, figsize=(15, 4))
for ax, im, t in zip(
    axes,
    [r_simple, r_crop, r_letter, x_show],
    ["resize directo\n(deforma)", "center crop\n(recorta)", "letterbox\n(rellena gris)",
     f"normalizado ImageNet\nrange aprox [{x_norm.min():.1f}, {x_norm.max():.1f}]"],
):
    ax.imshow(im); ax.set_title(t); ax.axis("off")
plt.tight_layout()
plt.show()

print(f"x_norm shape: {x_norm.shape}, dtype: {x_norm.dtype}")
print(f"Media por canal tras normalizar: {x_norm.mean(axis=(0,1))}")
print(f"Std   por canal tras normalizar: {x_norm.std(axis=(0,1))}")


## 7. Aumentación geométrica — transformaciones afines

Una transformación afín se describe con una matriz $2 \times 3$:

$$\begin{pmatrix} x' \\ y' \end{pmatrix} = \begin{pmatrix} a & b \\ c & d \end{pmatrix}\begin{pmatrix} x \\ y \end{pmatrix} + \begin{pmatrix} t_x \\ t_y \end{pmatrix}$$

Una **rotación + escala alrededor del centro** $(c_x, c_y)$ por ángulo $\theta$ y factor $s$ es:

$$M = \begin{pmatrix} s\cos\theta & s\sin\theta & (1 - s\cos\theta)c_x - s\sin\theta \cdot c_y \\ -s\sin\theta & s\cos\theta & s\sin\theta \cdot c_x + (1 - s\cos\theta)c_y \end{pmatrix}$$

`cv2.getRotationMatrix2D(center, theta, s)` la construye por ti. Después se aplica con `cv2.warpAffine`.

**Por qué aumentar geométricamente.** Forzamos al modelo a aprender **invarianzas** (a rotación, escala, traslación) que las clases reales sí cumplen.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider, Checkbox

img_bgr = cv2.imread("tigre.png")
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
h, w = img_rgb.shape[:2]
center = (w / 2.0, h / 2.0)


def afin(theta_deg, scale, tx, ty, flip_h):
    M = cv2.getRotationMatrix2D(center, theta_deg, scale)
    M[0, 2] += tx
    M[1, 2] += ty
    out = cv2.warpAffine(img_rgb, M, (w, h),
                         flags=cv2.INTER_LINEAR,
                         borderMode=cv2.BORDER_REFLECT_101)
    if flip_h:
        out = cv2.flip(out, 1)
    fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
    ax[0].imshow(img_rgb); ax[0].set_title("Original"); ax[0].axis("off")
    ax[1].imshow(out)
    ax[1].set_title(f"theta={theta_deg:.0f}° scale={scale:.2f} tx={tx} ty={ty} flip={flip_h}")
    ax[1].axis("off")
    plt.tight_layout(); plt.show()


interact(afin,
         theta_deg=FloatSlider(min=-90, max=90, step=5, value=15, description="rot (°)"),
         scale=FloatSlider(min=0.5, max=1.5, step=0.05, value=1.0, description="escala"),
         tx=IntSlider(min=-200, max=200, step=10, value=0, description="tx (px)"),
         ty=IntSlider(min=-200, max=200, step=10, value=0, description="ty (px)"),
         flip_h=Checkbox(value=False, description="Flip horizontal"));


## 8. Aumentación fotométrica — apariencia sin mover píxeles

Las transformaciones fotométricas alteran **el valor** de los píxeles, no su posición. Modelan variaciones del mundo real:

- **Brillo** → cambia $V$ en HSV: $V' = \text{clip}(V \cdot \beta)$.
- **Contraste** → afín en intensidad: $x' = \alpha(x - 127.5) + 127.5$.
- **Saturación** → cambia $S$ en HSV.
- **Matiz (Hue)** → desplaza $H$ módulo 180.
- **Ruido gaussiano** → simula sensor en baja luz: $x' = x + \mathcal{N}(0, \sigma^2)$.

**Buena práctica:** combina geométrica + fotométrica con probabilidades moderadas ($p \in [0.3, 0.6]$); aplicar todo con $p=1$ degrada la convergencia.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider

img_bgr = cv2.imread("tigre.png")
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)


def fotometrico(brillo, contraste, saturacion, hue_shift, sigma_ruido):
    # Brillo + saturación en HSV
    hsv = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV).astype(np.float32)
    hsv[..., 2] = np.clip(hsv[..., 2] * brillo, 0, 255)
    hsv[..., 1] = np.clip(hsv[..., 1] * saturacion, 0, 255)
    # Hue: desplazamiento módulo 180
    h = hsv[..., 0].astype(np.int16)
    h = (h + int(round(hue_shift / 2.0))) % 180
    hsv[..., 0] = h.astype(np.float32)
    rgb = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2RGB)
    # Contraste
    rgb_f = rgb.astype(np.float32)
    rgb = np.clip(contraste * (rgb_f - 127.5) + 127.5, 0, 255).astype(np.uint8)
    # Ruido gaussiano
    if sigma_ruido > 0:
        x01 = rgb.astype(np.float32) / 255.0
        x01 += np.random.normal(0, sigma_ruido, size=x01.shape).astype(np.float32)
        rgb = (np.clip(x01, 0, 1) * 255).astype(np.uint8)
    fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
    ax[0].imshow(img_rgb); ax[0].set_title("Original"); ax[0].axis("off")
    ax[1].imshow(rgb)
    ax[1].set_title(f"brillo={brillo:.2f}  contraste={contraste:.2f}  sat={saturacion:.2f}  "
                    f"hue={hue_shift}°  sigma={sigma_ruido:.02f}")
    ax[1].axis("off")
    plt.tight_layout(); plt.show()


interact(fotometrico,
         brillo=FloatSlider(min=0.3, max=2.0, step=0.05, value=1.0, description="brillo"),
         contraste=FloatSlider(min=0.3, max=2.0, step=0.05, value=1.0, description="contraste"),
         saturacion=FloatSlider(min=0.0, max=2.5, step=0.05, value=1.0, description="saturación"),
         hue_shift=IntSlider(min=-45, max=45, step=5, value=0, description="hue (°)"),
         sigma_ruido=FloatSlider(min=0.0, max=0.15, step=0.01, value=0.0, description="sigma"));


## 9. Modelos de ruido y denoising con métricas

Tres modelos de ruido típicos:

| Ruido | Modelo | Origen físico |
|---|---|---|
| **Gaussiano** | $y = x + \mathcal{N}(0, \sigma^2)$ | Sensor CMOS en baja luz. |
| **Sal y pimienta** | Píxeles aleatorios $\to 0$ o $255$ | Errores de transmisión, sensores defectuosos. |
| **Poisson (shot)** | $y \sim \mathrm{Poisson}(\lambda x)$ | Conteo discreto de fotones. |

Filtros y cuándo funcionan:

- **`GaussianBlur`** — lineal, **borra bordes**. Bueno como pre-paso de detección de bordes.
- **`medianBlur`** — no lineal, **elimina outliers** → mejor para sal y pimienta.
- **`bilateralFilter`** — pondera por proximidad espacial **y** similitud de color → suaviza preservando bordes.

Para comparar objetivamente usamos **PSNR**:

$$\mathrm{PSNR}(x, \hat{x}) = 10 \log_{10}\!\left(\frac{255^2}{\mathrm{MSE}(x, \hat{x})}\right) \;\; [\text{dB}]$$

Más alto = mejor reconstrucción. Valores típicos: $> 30$ dB calidad buena, $> 40$ dB casi imperceptible.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

img_bgr = cv2.imread("tigre.png")
img_bgr = cv2.resize(img_bgr, (320, 320), interpolation=cv2.INTER_AREA)
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
clean = img_rgb.astype(np.float32)

rng = np.random.default_rng(0)

# Ruido gaussiano
sigma = 25.0
noisy_g = np.clip(clean + rng.normal(0, sigma, clean.shape), 0, 255).astype(np.uint8)

# Ruido sal y pimienta
sp = img_rgb.copy()
p = 0.04
mask = rng.random(sp.shape[:2])
sp[mask < p / 2] = 0
sp[mask > 1 - p / 2] = 255

def psnr(x, y):
    mse = np.mean((x.astype(np.float32) - y.astype(np.float32)) ** 2)
    if mse == 0:
        return float("inf")
    return 10 * np.log10(255.0 ** 2 / mse)


filtros = [
    ("GaussianBlur 5x5 sigma=1.2", lambda im: cv2.GaussianBlur(im, (5, 5), 1.2)),
    ("medianBlur 5",              lambda im: cv2.medianBlur(im, 5)),
    ("bilateralFilter 9/75/75",    lambda im: cv2.bilateralFilter(im, 9, 75, 75)),
]

print("PSNR (dB) — más alto es mejor reconstrucción\n")
print(f"{'Filtro':<32} {'Ruido Gaussiano':>18} {'Sal y pimienta':>18}")
print("-" * 70)
filas_g, filas_sp = [], []
for nombre, fn in filtros:
    rg = fn(noisy_g); rsp = fn(sp)
    g = psnr(img_rgb, rg); s = psnr(img_rgb, rsp)
    print(f"{nombre:<32} {g:>18.2f} {s:>18.2f}")
    filas_g.append((nombre, rg)); filas_sp.append((nombre, rsp))

print(f"\nReferencia (sin filtrar): Gauss={psnr(img_rgb, noisy_g):.2f} dB  "
      f"SP={psnr(img_rgb, sp):.2f} dB")

# Mostrar resultados visualmente
fig, axes = plt.subplots(2, 4, figsize=(15, 7))
axes[0, 0].imshow(noisy_g); axes[0, 0].set_title(f"Ruido gaussiano\n({psnr(img_rgb, noisy_g):.1f} dB)"); axes[0, 0].axis("off")
for i, (n, im) in enumerate(filas_g):
    axes[0, i + 1].imshow(im); axes[0, i + 1].set_title(f"{n}\n({psnr(img_rgb, im):.1f} dB)"); axes[0, i + 1].axis("off")
axes[1, 0].imshow(sp); axes[1, 0].set_title(f"Sal y pimienta\n({psnr(img_rgb, sp):.1f} dB)"); axes[1, 0].axis("off")
for i, (n, im) in enumerate(filas_sp):
    axes[1, i + 1].imshow(im); axes[1, i + 1].set_title(f"{n}\n({psnr(img_rgb, im):.1f} dB)"); axes[1, i + 1].axis("off")
plt.tight_layout(); plt.show()


## 10. Pipeline profesional con Albumentations

En producción NUNCA se programan los aumentos a mano. Se usa una librería como **Albumentations** que:

- Aplica cada transformación con **probabilidad `p`** independiente.
- Está **optimizada en C++** (más rápida que `torchvision.transforms`).
- Soporta **bounding boxes**, **máscaras de segmentación** y **keypoints** de forma sincronizada.
- Se integra directamente con PyTorch (`ToTensorV2`) y TensorFlow.

> ⚠️ **Versiones.** En Albumentations 2.x cambiaron varias APIs:
> - `ShiftScaleRotate` → `A.Affine(translate_percent=..., scale=..., rotate=...)`
> - `GaussNoise(var_limit=...)` → `GaussNoise(std_range=(min, max))` (en $[0,1]$)
> - `RandomResizedCrop(height, width, ...)` → `RandomResizedCrop(size=(h, w), ...)`

El bloque siguiente genera una **cuadrícula 3×3 de variaciones aleatorias** de la misma imagen, simulando un mini-batch de entrenamiento.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import albumentations as A

print(f"Albumentations versión: {A.__version__}")

img_bgr = cv2.imread("tigre.png")
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

H, W = 224, 224

# Pipeline compatible con Albumentations 2.x
transform = A.Compose([
    A.RandomResizedCrop(size=(H, W), scale=(0.7, 1.0), ratio=(0.75, 1.33), p=1.0),
    A.HorizontalFlip(p=0.5),
    A.Affine(translate_percent=(-0.06, 0.06),
             scale=(0.85, 1.15),
             rotate=(-25, 25),
             border_mode=cv2.BORDER_REFLECT_101,
             p=0.7),
    A.RandomBrightnessContrast(brightness_limit=0.25, contrast_limit=0.25, p=0.6),
    A.HueSaturationValue(hue_shift_limit=12, sat_shift_limit=25, val_shift_limit=12, p=0.5),
    A.GaussNoise(std_range=(0.02, 0.08), p=0.3),
    A.MotionBlur(blur_limit=5, p=0.15),
], p=1.0)

# Generar 9 variaciones reproducibles
np.random.seed(42)
fig, axes = plt.subplots(3, 3, figsize=(11, 11))
for ax in axes.flat:
    out = transform(image=img_rgb)["image"]
    ax.imshow(out)
    ax.axis("off")
plt.suptitle("9 variaciones aleatorias del mismo origen — simula un mini-batch", y=1.00, fontsize=13)
plt.tight_layout(); plt.show()


## 11. Estadísticas de un mini-dataset (media / std por canal)

Cuando entrenas con datos propios (no ImageNet) debes calcular tu propia media y desviación estándar **por canal** sobre el dataset de entrenamiento, así:

$$\mu_c = \frac{1}{N H W}\sum_{n=1}^{N}\sum_{y,x} x^{(n)}_{c,y,x}, \quad \sigma_c^2 = \frac{1}{N H W}\sum_{n,y,x}\big(x^{(n)}_{c,y,x} - \mu_c\big)^2$$

Para datasets grandes se calcula de forma **incremental** (Welford) o por sumas y sumas de cuadrados parciales para evitar overflow.

A continuación simulamos un mini-dataset generando 16 versiones aumentadas de la misma imagen y calculamos la estadística.

In [ ]:
import cv2
import numpy as np
import albumentations as A

img_bgr = cv2.imread("tigre.png")
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

augment = A.Compose([
    A.RandomResizedCrop(size=(224, 224), scale=(0.7, 1.0), p=1.0),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.7),
])

N = 16
# Acumuladores tipo Welford simplificado
suma = np.zeros(3, dtype=np.float64)
suma_cuad = np.zeros(3, dtype=np.float64)
total_px = 0

np.random.seed(0)
for _ in range(N):
    x = augment(image=img_rgb)["image"].astype(np.float64) / 255.0
    suma += x.sum(axis=(0, 1))
    suma_cuad += (x ** 2).sum(axis=(0, 1))
    total_px += x.shape[0] * x.shape[1]

mean = suma / total_px
var = suma_cuad / total_px - mean ** 2
std = np.sqrt(var)

print(f"Mini-dataset simulado: N={N} imágenes 224x224")
print(f"Media por canal (R,G,B): {mean}")
print(f"Std   por canal (R,G,B): {std}")
print()
print("Referencia ImageNet:")
print("  Media: [0.485, 0.456, 0.406]")
print("  Std:   [0.229, 0.224, 0.225]")


## 12. Ejercicios propuestos

Para fijar los conceptos, intenta lo siguiente (añade tus celdas debajo):

1. **Segmentación robusta.** Usa el widget de la sección 5 para encontrar un rango HSV que aísle SOLO el ojo del tigre. Aplica luego `cv2.findContours` para dibujar el contorno detectado.

2. **Composición de transformaciones afines.** Encuentra una matriz $M$ que **rote 30°, escale 0.8× y traslade (50, -30) px** **en una sola** `warpAffine`. Verifica que el resultado coincide con aplicar las tres operaciones por separado.

3. **CLAHE vs ecualización global.** Carga una imagen con **iluminación muy desigual** (busca una foto a contraluz). Compara cuantitativamente la mejora de contraste usando el **rango intercuartil** de la luminancia $Y$ antes y después de cada método.

4. **PSNR vs SSIM.** Implementa el cálculo de **SSIM** (`from skimage.metrics import structural_similarity`) y reproduce la tabla de la sección 9 con ambas métricas. ¿En qué casos discrepan PSNR y SSIM?

5. **Pipeline de tu propio dataset.** Toma 10 imágenes propias, calcula media/std por canal con el método de la sección 11 y compara con las estadísticas de ImageNet. ¿Qué tan distintas son? ¿Cómo afectaría usar las equivocadas?

6. **(Avanzado) Test-time augmentation (TTA).** Define un pipeline determinista de 5 aumentaciones (sin aleatoriedad), aplícalas a una imagen y promedia "predicciones" (aquí: el histograma del canal V). Discute por qué TTA mejora la robustez en inferencia.

---

### Lecturas recomendadas

- Szeliski, R. *Computer Vision: Algorithms and Applications* (2nd ed.) — capítulos 3 y 4.
- Goodfellow, Bengio, Courville. *Deep Learning*, cap. 9 (convolución).
- Buslaev et al. (2020). *Albumentations: Fast and Flexible Image Augmentations*. *Information* 11(2):125.
- Documentación oficial: https://albumentations.ai/docs/